# Word and Sentence Embeddings


## Feed-forward neural networks

<img class="lecture-figure figure-48" src="../img/mlp.svg" alt="A feed-forward neural network with an input layer, hidden layers and an output layer">

The network needs numerical inputs. **What numbers should represent a word?**

Use the familiar network to reconnect with the previous lecture. The focus now is the representation supplied to the input layer. Return to this diagram when the word-prediction network is introduced: its hidden representation will become the word embedding.

## Representing words

A representation maps each word to a vector.

What should the vectors for **apple**, **orange** and **rabbit** look like?

We want a model to distinguish words and make use of similarities between them.

## One-hot word representations

For the vocabulary $V=\{\text{apple},\text{orange},\text{rabbit}\}$:

$$\begin{aligned}
f(\text{apple})&=(1,0,0)\\
f(\text{orange})&=(0,1,0)\\
f(\text{rabbit})&=(0,0,1)
\end{aligned}$$

A **one-hot vector** has one coordinate per vocabulary item.

## One-hot vectors are orthogonal

<img class="lecture-figure figure-45" src="../img/sparse_binary.svg" alt="The original three-dimensional visualization: apple, orange and rabbit lie on three perpendicular unit axes.">

Every pair of different words has dot product zero. The representation gives **apple–orange** no more similarity than **apple–rabbit**.

Word identity is represented exactly, but the geometry supplies no graded similarity. Sparse storage avoids materializing every zero; the problem here is the information represented, not a claim that storing one-hot vectors must be expensive.

## Cosine similarity

Cosine compares vector directions:

$$\cos(u,v)=\frac{u\cdot v}{\lVert u\rVert\,\lVert v\rVert}$$

Same direction: **1**. Orthogonal: **0**. Opposite direction: **−1**.

What happens when vectors can point between the axes?

Cosine is undefined for a zero vector. SciPy calls 1 minus cosine similarity cosine distance. The quiz next isolates direction from length and from distance between endpoints.

## Cosine similarity

<div class="cosine-quiz">
<div><img class="lecture-figure figure-44" src="../img/word2vec_2027/cosine_quiz.svg" alt="Query q=(1,1); A=(3,3), B=(1,0.4), C=(5,-1), D=(-1,-1)."></div>
<div>
<p><b>Which vector has the highest cosine similarity to <i>q</i>?</b></p>
<p class="fragment"><b>A:</b> same direction as <i>q</i>, so cosine = 1.</p>
<p class="fragment">If we multiply <b>B</b> by 10, does the answer change?</p>
<p class="fragment"><b>No.</b> Positive scaling preserves direction.</p>
</div>
</div>

Take answers before each reveal. The other cosines are B=0.919145, C=0.554700 and D=-1. In the original plot B is closest to q by Euclidean distance, while C has the largest norm. Neither wins by cosine. Multiplying B by ten changes its norm and endpoint distance but preserves its cosine. The figure uses the original coordinates from the earlier quiz.

## Dense word embeddings

<img class="lecture-figure figure-42" src="../img/dense_continuous.svg" alt="The original two-dimensional illustration: apple and orange point in similar directions, while rabbit points in a different direction.">

A **dense word embedding** gives each word a short vector of real-valued features.

**How could we learn a useful geometry like this from text?**

This is an illustrative geometry, not a measured training result. Dense describes a representation, not a particular training algorithm. A learned feature can be shared across many words, unlike a separate one-hot coordinate for every word. Later, compare this goal with what context counts already achieve.

## The distributional hypothesis

Words that occur in similar contexts often have related meanings.

> You shall know a word by the company it keeps.

Firth (1957)

We can turn this idea into a representation by **counting nearby words**.

This motivates the next matrix directly. Context can capture syntactic as well as semantic behaviour. Corpus choice and social patterns shape the resulting similarities, so distributional similarity is not a complete account of meaning.

## Counting context words

<div class="lecture-columns">
<div>
<p>Peel the <b>apple</b>.<br>Slice the <b>apple</b>.</p>
<p>Peel the <b>orange</b>.<br>Slice the <b>orange</b>.</p>
<p>Feed the <b>rabbit</b>.</p>
</div>
<div>
<p><b>Selected context counts</b></p>
<table><thead><tr><th>Word</th><th>peel</th><th>slice</th><th>feed</th></tr></thead>
<tbody><tr><td>apple</td><td>1</td><td>1</td><td>0</td></tr>
<tr><td>orange</td><td>1</td><td>1</td><td>0</td></tr>
<tr><td>rabbit</td><td>0</td><td>0</td><td>1</td></tr></tbody></table>
</div>
</div>

A row is a word vector. Its coordinates count words within a context window.

Lowercase and remove punctuation. Use a symmetric window of radius two within each sentence. The figure displays only the context columns peel, slice and feed. The full vocabulary also includes the three nouns and the; apple and orange have two occurrences of the nearby, while rabbit has one. Apple and orange still have identical full context-count rows. Explicitly identify what a row and a column represent before continuing.

## Similar contexts

**Apple** and **orange** never occur together in this corpus.

Does that prevent their context-count vectors from being similar?

**A.** Yes, they must occur near each other.  
**B.** No, they can share the same context words.

**B.** Both occur with **peel** and **slice**. Similarity can come from shared contexts, without direct co-occurrence.

Use this as a concept check. Ask students to explain the answer using the preceding rows and the earlier cosine definition. The displayed apple and orange rows have cosine one. Do not equate distributional similarity with how often the two words directly occur together.

## Counts and TF–IDF for documents

We can also represent a **document** by counting its words.

**Term frequency–inverse document frequency (TF–IDF)** downweights words found in many documents.

The features still correspond to vocabulary items: **apple** and **orange** occupy separate coordinates.

Can we give a model features that these related words can share?

Distinguish a word–context matrix, whose rows are words, from a document–term matrix, whose rows are documents. TF–IDF is a strong baseline for document classification and retrieval. It can capture overlap through the other words in a document, but it does not directly encode that apple and orange are related. The exact smoothed scikit-learn formula and row normalization are in the optional technical notes.

## Why use dense vectors?

For example, **50,000 context features** can become **300 learned features**.

- Related words can share features, helping a model generalize across words.
- A smaller input dimension makes neural models easier to work with.

Counts and TF–IDF remain useful baselines, especially when exact words matter.

The sizes 50,000 and 300 illustrate a typical scale rather than reporting an experiment. The motivation is feature sharing and a compact input dimension, not a universal storage or accuracy advantage: sparse counts can be very efficient. Dense methods compress or learn recurring context patterns, potentially smoothing across correlated or sparse observations. This may lose information and must be evaluated for the task. Weighted-count compression is another route to dense vectors; PPMI and SVD are covered only in the optional notes.

## Learning a representation through prediction

The feed-forward network can learn features that help it predict a word from its context.

> Peel the **_____**.

The original text, “Peel the **apple**”, supplies the word to predict. **The corpus supplies the training targets.**

Connect this explicitly to the opening network: the numerical representation inside the network is useful because it helps the prediction task. Apple and orange can both fit this context, but each corpus occurrence supplies a concrete training target. Learning over many contexts can make their representations similar. This is the same self-supervised idea as language modelling, here used to obtain reusable word vectors.

## Continuous bag-of-words (CBOW)

Average the vectors of the context words and predict the missing word.

$$\underbrace{\frac{f(\text{peel})+f(\text{the})}{2}}_{\text{context representation}}\quad\longrightarrow\quad\text{predict apple}$$

Training these predictions also learns the word vectors being averaged.

CBOW is the name of a training architecture, not a generic name for every average of word vectors. We use the mean-pooling variant. The context window normally includes words on both sides when available; apple is at the end of this example sentence. The original architecture diagram sometimes labels the projection as SUM. Source: Mikolov et al. (2013), Efficient Estimation of Word Representations in Vector Space, https://arxiv.org/abs/1301.3781.

## Word2vec: CBOW and skip-gram

<img class="lecture-figure figure-40" src="../img/word2vec_2027/cbow_skipgram.svg" alt="CBOW averages the context vectors for peel and the to predict apple. Skip-gram uses apple to predict peel and the separately.">

**Word2vec** includes both architectures. We will look inside **skip-gram**, which predicts nearby words from one word.

Both architectures learn static word embeddings. The corpus signal is the same co-occurrence evidence used earlier, but here it supplies prediction examples. Source: Mikolov et al. (2013), https://arxiv.org/abs/1301.3781.

## Inside skip-gram

<img class="lecture-figure figure-44" src="../img/word2vec_2027/network.svg" alt="A one-hot input selects apple. Input-to-hidden weights produce a small linear embedding layer. Hidden-to-output weights score context words, with peel highlighted as an observed context.">

The **input weights** contain a vector for each word. The **output weights** turn that vector into context predictions.

Relate the layers directly to the original feed-forward network. For vocabulary size |V| and embedding dimension d, the input weights can be stored as W_in of shape |V| by d, with one row per word. Multiplication by a one-hot input selects its row. The output weights W_out have shape d by |V| and map that vector to scores over context words; softmax converts the scores to probabilities in the full-softmax exposition. Unlike a conventional multilayer perceptron, this projection has no nonlinear hidden activation. Both sets of weights are trainable parameters. A common downstream convention keeps the input vectors. The drawing shows selected vocabulary items and three hidden coordinates for readability.

## Initialization and training

<img class="lecture-figure figure-43" src="../img/word2vec_2027/training.svg" alt="Word vectors start as small random numbers. A word and its vector feed a context prediction, which is compared with the observed context. Backpropagation and stochastic gradient descent update the network weights.">

Start with **small random word vectors**. Repeated predictions and **stochastic gradient descent (SGD)** updates make them useful for predicting context.

Use the diagram to identify initialization, a forward prediction, prediction error, backpropagation and the weight update. This is an overview, not a derivation of SGD. The learned word vectors are network parameters, so they change as the loss decreases. Word2vec does not initialize them from a co-occurrence matrix. The original C implementation initializes the input weights randomly and the output weights to zero; other implementations can choose differently. This implementation detail and a worked update are in the optional notes. For large vocabularies, negative sampling replaces full softmax with observed-pair versus noise-pair classification to reduce computation. Source: https://github.com/tmikolov/word2vec/blob/master/word2vec.c and https://arxiv.org/abs/1310.4546.

## Learned word embeddings

<img class="lecture-figure figure-48" src="../img/word_representations.svg" alt="The original projection of a learned word embedding space, with clusters of words that share syntactic or semantic behaviour.">

Words with similar contexts can acquire similar vectors. **We can reuse these vectors as inputs to other models.**

This closes the loop with the opening question about neural-network inputs and the earlier illustrative dense geometry. The figure is a two-dimensional t-distributed stochastic neighbor embedding (t-SNE) projection of higher-dimensional vectors. Projection can distort distances; similarity claims should be checked in the original space. Word vectors often reflect syntactic as well as semantic patterns. Different word senses and biases in the training corpus remain limitations.

## How counts and prediction are connected

Both use evidence about **which words occur near one another**.

- **Count-based methods** can produce dense vectors by fitting or compressing co-occurrence statistics. **Global Vectors (GloVe)** is one example.
- **Word2vec** learns dense vectors by optimizing context predictions.

The shared evidence connects them. Their different objectives can produce different vectors.

Do not present dense versus sparse as synonymous with prediction-based versus count-based. GloVe fits weighted log co-occurrence counts with learned vectors and bias terms. PPMI plus truncated SVD is another count-based route. Some prediction objectives have a precise matrix-factorization interpretation under stated assumptions, but word2vec is not simply SVD applied to raw counts. The optional notes explain that connection rather than introducing its terminology into the core slides. Sources: https://aclanthology.org/D14-1162/ ; https://papers.nips.cc/paper_files/paper/2014/file/b78666971ceae55a8e87efb7cbfd9ad4-Paper.pdf ; https://aclanthology.org/Q15-1016/.

## Representing a whole sentence

We now have one vector per word. A classifier may need one vector for the **whole sentence**.

**Mean pooling of static word embeddings** averages its word vectors:

$$f(s)=\frac{1}{n}\sum_{i=1}^{n}f(w_i)$$

The result is a **sentence embedding** with the same dimension as the word vectors.

This is the transition from word representations to sentence representations. Static means the same word vector is used in every sentence. Mean pooling has no trainable parameters of its own. It can combine vectors learned with CBOW, skip-gram, GloVe or other methods.

## Averaging during training and after training

**CBOW training:** average a context window to predict a word. Prediction errors update the word vectors.

**Sentence mean pooling:** average already learned vectors across a complete sentence.

The averaging operation is the same. What we represent and what we train are different.

Use the earlier “peel the ___” example to contrast averaging just the context with averaging the complete sentence. The goal is to prevent calling every averaged sentence representation CBOW. No discussion of Sentence-BERT or Transformer architectures is needed here.

## Who chases whom?

<div class="sentence-pair"><p><b>dogs</b> chase cats</p><p><b>cats</b> chase dogs</p></div>

With the same static word vectors, will these sentences have **the same mean-pooled vector or different vectors?**

**The same vector:**

$$\frac{f(\text{dogs})+f(\text{chase})+f(\text{cats})}{3}
=\frac{f(\text{cats})+f(\text{chase})+f(\text{dogs})}{3}$$

The meaning changes, but the average loses word order.

Let students answer before revealing the sum. This is the second new concept quiz. The same multiset of word tokens gives the same mean. The following code provides a concrete numerical example after the conceptual answer.

In [ ]:
import numpy as np

word_vectors = {
    "dogs": [1., 0.], "chase": [0., 1.], "cats": [.8, .2]
}

def mean_pool(sentence):
    return np.mean([word_vectors[w] for w in sentence.split()], axis=0)

## Mean pooling in code

Each word has a vector. Averaging ignores the order in which we add them.

In [ ]:
for sentence in ("dogs chase cats", "cats chase dogs"):
    print(sentence, mean_pool(sentence))

dogs chase cats [0.6 0.4]
cats chase dogs [0.6 0.4]


The preceding setup cell is skipped in the slideshow and defines the toy word vectors and mean_pool. It remains available in normal notebook view. Run it before rerunning this short demonstration. The cached output lets the slide be presented without executing code.

## Mean pooling in the library

The **Sentence Transformers library** can average static GloVe word embeddings:

```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/average_word_embeddings_glove.6B.300d"
)
vectors = model.encode(["dogs chase cats", "cats chase dogs"])
```

This model contains a word-vector lookup and mean pooling. Each sentence gets a 300-dimensional vector.

SentenceTransformer is the library interface. This particular model has no Transformer layers; it uses static GloVe vectors and mean pooling. Calling it a sentence transformer would confuse the library name with the model architecture. Model documentation: https://huggingface.co/sentence-transformers/average_word_embeddings_glove.6B.300d . Running this optional example requires the sentence-transformers package and a model download.

## What is still missing?

The word **bank** has the same static vector in “river bank” and “bank loan”.

Mean pooling also loses **word order**, as “dogs chase cats” showed.

Next: models that use the sequence of words and build representations in context.

Keep the distinction precise: the word vector for bank is unchanged, but the average of the entire sentence can still differ because the other words differ. The next lectures develop sequence models and contextual representations. Do not claim that mean pooling makes every sentence containing a polysemous word indistinguishable.

## Summary

- One-hot vectors distinguish words but give no graded similarity.
- Context counts reveal shared usage. Dense vectors can encode useful patterns in fewer features.
- A prediction network learns word embeddings as part of its weights.
- Mean pooling gives a sentence vector, while losing word order.

## Further reading

- Jurafsky & Martin: [Embeddings](https://web.stanford.edu/~jurafsky/slp3/5.pdf)
- Mikolov et al. (2013): [Word2vec architectures](https://arxiv.org/abs/1301.3781)
- Pennington et al. (2014): [GloVe](https://aclanthology.org/D14-1162/)
- [Optional technical notes](dl-representations_notes.ipynb): count weighting, compression and training details